# Handwritten Digit Recognition with HOG and SVM

Real handwritten digits are included as image files.

## Step 1: Import libraries

HOG extracts stroke direction.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from skimage.feature import hog
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## Step 2: Load digit folders

Folders 0–9 are class labels.

In [ ]:
DATASET_DIR=Path("../datasets/10_handwritten_digits")
X=[]; labels=[]
for folder in sorted(DATASET_DIR.iterdir(),key=lambda p:int(p.name)):
 for fp in sorted(folder.glob("*.png")):
  X.append(np.array(Image.open(fp).convert("L"))); labels.append(folder.name)
images=np.array(X); labels=np.array(labels)
print(images.shape)

## Step 3: Extract HOG features

HOG captures digit strokes.

In [ ]:
features=np.array([hog(im/255.0,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2)) for im in images])

## Step 4: Train and evaluate

SVM performs ten-class classification.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(features,labels,test_size=.25,random_state=42,stratify=labels)
model=make_pipeline(StandardScaler(),SVC(C=5)).fit(X_train,y_train)
pred=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred); plt.show()

# Test One Single Handwritten Digit Image

This section loads one individual handwritten digit image, extracts HOG features and predicts the digit using the trained SVM model.

The new image is converted to grayscale and resized to 64 × 64, matching the training process.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog

single_image_path = Path(
    "../datasets/10_handwritten_digits/7/000.png"
)

single_image = Image.open(single_image_path).convert("L")
single_image = single_image.resize((64, 64))
single_image_array = np.array(single_image)

single_features = hog(
    single_image_array / 255.0,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
)

single_features_2d = single_features.reshape(1, -1)
predicted_digit = model.predict(single_features_2d)[0]

print("Image shape:", single_image_array.shape)
print("HOG feature shape:", single_features.shape)
print("Predicted digit:", predicted_digit)

plt.figure(figsize=(4, 4))
plt.imshow(single_image_array, cmap="gray")
plt.title(f"Predicted Digit: {predicted_digit}")
plt.axis("off")
plt.show()
